# Wind outage: key data visuals

**TL;DR**
- **Neighbours are a good proxy:** 10-min correlation of T with any neighbour is 0.66–0.80 (best: 0.80); averaged over a day it reaches 0.97.
- **Timing hardly matters:** best time shift is +20 min (H) and 0 min (O); a shift gains only 0.011 in r.
- **Energy comes from the steep part of the power curve:** 97% of energy at 3–11.5 m/s; 1 m/s of error costs about 440 kW per turbine at the median 5.3 m/s.
- **Simple baseline:** T = a + b·H + c·O has a wind speed RMSE of 1.40 m/s and a power RMSE of 878 kW on 4–9 Sep; total energy of T1–T3 is within 0.1% only because errors cancel; single turbines are off by -8% to +19%.
- **Which inputs matter:** neighbour wind speed (|r| 0.80–0.81); direction, hour, temperature, pressure and stddev add at most |r| 0.08 on the model residual; hour of day (sin 0.07, cos 0.03) is the most promising time-based correction (T/H 0.85 at night, 1.00 at 15:00).
- **Data is clean enough:** 95.5–96.9% valid wind speed; gaps are NaN, never zero.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
from preprocessing import (load_clean, to_wide, load_power_curve, wind_to_power, energy_kwh,
                           quality_report, TARGETS, REFERENCES, STEP_HOURS)

pio.renderers.default = "plotly_mimetype+notebook_connected"

long = load_clean()
ws = to_wide(long)                       # timestamp x turbine, wind speed [m/s]
wd = to_wide(long, "wind_direction")
temp = to_wide(long, "ambient_temperature")
pres = to_wide(long, "air_pressure")
sd = to_wide(long, "wind_speed_stddev")
K = {}                                   # key numbers, reused in the text

In [2]:
H_COLS, O_COLS = ["H1", "H2", "H3", "H4", "H5"], ["O1", "O2"]
split = long[long.turbine_name == "T1"].set_index("timestamp")["split"]
train, val, known = split == "train", split == "val", split != "hidden"

# park means: average of the turbines that report in each 10-min step
park = pd.DataFrame({"T": ws[TARGETS].mean(axis=1),
                     "H": ws[H_COLS].mean(axis=1),
                     "O": ws[O_COLS].mean(axis=1)})

def circ_mean(deg):
    """Circular mean of wind directions [deg], row-wise."""
    r = np.deg2rad(deg)
    return np.rad2deg(np.arctan2(np.sin(r).mean(axis=1), np.cos(r).mean(axis=1))) % 360

dir_T, dir_H = circ_mean(wd[TARGETS]), circ_mean(wd[H_COLS])
power_curve = load_power_curve("T1")

In [3]:
COL = {"T": "#2a78d6", "H": "#eb6834", "O": "#1baf7a", "grey": "#8a8984", "ink": "#1d1d1b", "T_light": "#9ec3ee"}
BLUES = [[0, "#f2f7fd"], [1, "#104281"]]
HIDDEN, VAL = ("2026-09-01", "2026-09-04"), ("2026-09-04", "2026-09-10")

def style(fig, title, subtitle, height=470):
    """Shared look: conclusion title, grey subtitle, white template, hairline grid."""
    n_sub = subtitle.count("<br>") + 1
    fig.update_layout(
        template="plotly_white", font=dict(size=13), height=height,
        title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:{COL['grey']}'>{subtitle}</span>",
                   x=0, xanchor="left", yref="container", y=0.97, yanchor="top"),
        margin=dict(t=70 + 20 * n_sub, l=70, r=30, b=60))
    grid = dict(showgrid=True, gridcolor="#ececea", gridwidth=0.5, zeroline=False)
    fig.update_xaxes(**grid)
    fig.update_yaxes(**grid)
    return fig

def mark_windows(fig):
    """Light grey bands for the hidden test and validation windows (time charts)."""
    for (a, b), name, op in ((HIDDEN, "hidden test", 0.16), (VAL, "validation", 0.08)):
        fig.add_vrect(x0=a, x1=b, fillcolor=COL["grey"], opacity=op, line_width=0, layer="below")
    fig.add_annotation(x=VAL[0], xanchor="right", y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                       text="hidden test", font=dict(size=11, color=COL["grey"]))
    fig.add_annotation(x=VAL[0], xanchor="left", y=1.0, yref="paper", yanchor="bottom", showarrow=False, yshift=14,
                       text="validation", font=dict(size=11, color=COL["grey"]))

### 1. Data quality: small gaps, nothing hidden as zero
**Takeaway:** Every turbine has 95.5–96.9% valid wind speed; the big white blocks are the absent 3–4 Aug rows and the hidden 1–3 Sep window, plus one 15.7 h H1 outage.

**For the model:** Keep gaps as NaN; fill only short reference gaps (20 min or less) and never fill the target.

In [4]:
rep = quality_report(long)
K["valid_min"], K["valid_max"] = float(rep.valid_ws_pct.min()), float(rep.valid_ws_pct.max())
K["valid_mean"] = float(rep.valid_ws_pct.mean())
K["h1_gap_h"] = float(rep.loc["H1", "longest_sensor_gap_h"])
display(rep)

,rows_absent,nan_ws_in_files,sensor_dropouts,out_of_range,stuck_sensor,longest_sensor_gap_h,pressure_usable,stddev_usable,valid_ws_pct
turbine_name,,,,,,,,,
T1,288,156,3,0,0,5.0,True,True,96.5
T2,288,109,2,0,0,0.3,True,True,96.9
T3,288,111,4,0,0,0.3,True,True,96.9
H1,288,286,0,0,0,15.7,True,True,95.5
H2,288,116,0,0,0,0.3,True,True,96.8
H3,288,120,0,0,0,0.3,True,True,96.8
H4,288,145,0,0,0,0.2,True,True,96.6
H5,288,120,0,0,0,0.3,True,True,96.8
O1,288,131,0,0,0,0.3,False,False,96.7


In [5]:
daily = ws.notna().resample("1D").mean().mul(100).T            # turbine x day, % valid
days = daily.columns + pd.Timedelta(hours=12)                  # centre cells on their day
absent_days = [pd.Timestamp("2026-08-03"), pd.Timestamp("2026-08-04")]
h1_low = daily.loc["H1"][(daily.loc["H1"] < 50) & ~daily.columns.isin(absent_days)]
K["h1_gap_day"] = h1_low.index[0].strftime("%d %b")

fig = go.Figure(go.Heatmap(
    z=daily.values, x=days, y=daily.index, colorscale=BLUES, zmin=0, zmax=100, xgap=1, ygap=1,
    colorbar=dict(title="% valid", thickness=12),
    hovertemplate="%{y}, %{x|%d %b}: %{z:.0f}% valid<extra></extra>"))
for y in (2.5, 7.5):
    fig.add_hline(y=y, line_color=COL["grey"], line_width=1)
mark_windows(fig)
fig.add_annotation(x="2026-08-04", y="O2", text="3–4 Aug: rows absent<br>from the files", showarrow=True,
                   arrowhead=2, ax=0, ay=50, font=dict(size=12, color=COL["grey"]), arrowcolor=COL["grey"])
fig.add_annotation(x=days[daily.columns.get_loc(h1_low.index[0])], y="H1", text="H1 outage 13–15 Jul<br>(10 h + 16 h)",
                   showarrow=True, arrowhead=2, ax=0, ay=-120, font=dict(size=12, color=COL["grey"]), arrowcolor=COL["grey"])
fig.update_yaxes(autorange="reversed", title="Turbine (T, then H, then O)")
fig.update_xaxes(title="Day", tickformat="%d %b")
style(fig, f"Data is {K['valid_min']:.1f}–{K['valid_max']:.1f}% complete; gaps are empty, not zero",
      "Share of valid 10-min wind speed readings per turbine and day (white = no data)", height=560)
fig.update_layout(margin=dict(t=150, b=100))
fig.show()

### 2. Neighbouring parks move together
**Takeaway:** In the week 01 Jul–07 Jul the three park averages rise and fall almost together (T vs H r = 0.80), with T slightly lower.

**For the model:** Park averages are a stable, low-noise input for rebuilding T.

In [6]:
# pick the 7-day window (Jul or Aug, avoiding the absent rows) with the most varied H-park wind
starts = [s for s in pd.date_range("2026-07-01", "2026-08-25", freq="1D")
          if not (pd.Timestamp("2026-08-03") - pd.Timedelta(days=7) < s < pd.Timestamp("2026-08-05"))]
spread = {s: park.loc[s:s + pd.Timedelta(days=7) - pd.Timedelta(minutes=10), "H"].std() for s in starts}
w0 = max(spread, key=spread.get)
wk = park.loc[w0:w0 + pd.Timedelta(days=7) - pd.Timedelta(minutes=10)]
K["week"] = f"{w0:%d %b}–{(w0 + pd.Timedelta(days=6)):%d %b}"
K["week_r"] = float(wk["T"].corr(wk["H"]))
K["week_peak"] = float(wk["T"].max())

names = {"T": "Park T (T1–T3)", "H": "Park H (H1–H5)", "O": "Park O (O1–O2)"}
fig = go.Figure()
for p in "HOT":
    fig.add_trace(go.Scatter(x=wk.index, y=wk[p], name=names[p], mode="lines",
                             line=dict(color=COL[p], width=2 if p == "T" else 1.5),
                             hovertemplate="%{y:.1f} m/s"))
pk = wk["T"].idxmax()
fig.add_annotation(x=pk, y=wk["T"].max(), text=f"T peak {wk['T'].max():.1f} m/s", showarrow=True, ax=-50, ay=-45,
                   font=dict(size=12, color=COL["T"]), arrowcolor=COL["T"])
fig.update_layout(hovermode="x unified")
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.07), title="Time")
fig.update_yaxes(title="Wind speed [m/s]", rangemode="tozero")
style(fig, "Park T moves with its neighbours",
      f"10-min mean wind speed per park, {K['week']} (most varied week); correlation T vs H this week: {K['week_r']:.2f}",
      height=520)
fig.show()

### 3. Every neighbour is a usable reference
**Takeaway:** Correlation ranges 0.66–0.80; O parks (0.77 on average) beat H parks (0.72); inside park T the turbines agree more (T1–T2 0.83, T1–T3 0.84, T2–T3 0.90).

**For the model:** No single neighbour dominates, so combine several instead of picking one.

In [7]:
cm = ws.corr(min_periods=100)                       # pairwise on overlapping rows
C = cm.loc[TARGETS, REFERENCES]
K["r_max"], K["r_min"] = float(C.values.max()), float(C.values.min())
K["r_best_ref"] = C.max().idxmax()
K["r_best_t"] = C.max(axis=1).idxmax()
K["r_best"] = float(C.values.max())
K["r_H_mean"] = float(C[H_COLS].values.mean())
K["r_O_mean"] = float(C[O_COLS].values.mean())
K["r_T12"], K["r_T13"], K["r_T23"] = float(cm.loc["T1", "T2"]), float(cm.loc["T1", "T3"]), float(cm.loc["T2", "T3"])
zmin = round(float(C.values.min()) - 0.1, 1)

fig = go.Figure(go.Heatmap(z=C.values, x=C.columns, y=C.index, colorscale=BLUES, zmin=zmin, zmax=1, xgap=2, ygap=2,
                           colorbar=dict(title="r", thickness=12),
                           hovertemplate="%{y} vs %{x}: r = %{z:.2f}<extra></extra>"))
for i, t in enumerate(C.index):
    for j, r in enumerate(C.columns):
        v = C.iloc[i, j]
        fig.add_annotation(x=r, y=t, text=f"{v:.2f}", showarrow=False,
                           font=dict(color="white" if v > zmin + 0.5 * (1 - zmin) else COL["ink"], size=14))
fig.add_vline(x=4.5, line_color=COL["grey"], line_width=1)
fig.update_yaxes(autorange="reversed", title="Target turbine (park T)")
fig.update_xaxes(title="Reference turbine (H park, then O park)")
style(fig, f"Every neighbour tracks park T (r = {K['r_min']:.2f}–{K['r_max']:.2f})",
      f"Pearson correlation of 10-min wind speed; colour scale starts at {zmin}. Within park T: "
      f"T1–T2 {K['r_T12']:.2f}, T1–T3 {K['r_T13']:.2f}", height=400)
fig.show()

### 4. The relationship is roughly a straight line
**Takeaway:** H explains 51% (T1), 62% (T2) and 59% (T3) of T's 10-min variation; T runs below the 1:1 line (slope 0.71–0.82).

**For the model:** A linear fit with an intercept is a fair start; fit it per turbine, not with T = H.

In [8]:
H = park["H"]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=TARGETS)
xs = np.array([0, 18.0])
K["r2"], K["slope"], K["icpt"] = {}, {}, {}
for k, t in enumerate(TARGETS, start=1):
    d = pd.concat([H, ws[t]], axis=1, keys=["h", "t"]).dropna()
    b, a = np.polyfit(d.h, d.t, 1)
    r2 = float(np.corrcoef(d.h, d.t)[0, 1] ** 2)
    K["r2"][t], K["slope"][t], K["icpt"][t] = r2, float(b), float(a)
    fig.add_trace(go.Histogram2d(x=d.h, y=d.t, xbins=dict(start=0, end=18, size=0.5), ybins=dict(start=0, end=18, size=0.5),
                                 coloraxis="coloraxis", hovertemplate="H %{x} m/s, T %{y} m/s: %{z} samples<extra></extra>"),
                  row=1, col=k)
    fig.add_trace(go.Scatter(x=xs, y=xs, mode="lines", name="1:1 line (T = H)", legendgroup="1",
                             line=dict(color=COL["grey"], width=1.5), showlegend=(k == 1), hoverinfo="skip"), row=1, col=k)
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", name="Best straight-line fit", legendgroup="2",
                             line=dict(color=COL["ink"], width=2), showlegend=(k == 1),
                             hovertemplate="fit: %{y:.1f} m/s<extra></extra>"), row=1, col=k)
    fig.layout.annotations[k - 1].update(text=f"<b>{t}</b>  R² = {r2:.2f}")
fig.update_layout(coloraxis=dict(colorscale=BLUES, colorbar=dict(title="10-min<br>samples", thickness=12, len=0.5, y=0.3)))
fig.update_xaxes(title="H-park mean [m/s]", range=[0, 18])
fig.update_yaxes(range=[0, 18])
fig.update_yaxes(title="Wind speed at the T turbine [m/s]", col=1)
style(fig, f"A straight line captures about {np.mean(list(K['r2'].values())) * 100:.0f}% of T's 10-min variation",
      f"Each cell counts 10-min samples; R² = share of T variance explained by H mean "
      f"(T1 {K['r2']['T1']:.2f}, T2 {K['r2']['T2']:.2f}, T3 {K['r2']['T3']:.2f})", height=480)
fig.show()

### 5. Short gusts are noise, trends are predictable
**Takeaway:** Correlation T vs H rises from 0.80 (10 min) to 0.85 (1 h), 0.89 (3 h) and 0.97 (1 day); O behaves the same (0.81 to 0.96).

**For the model:** Expect good totals over hours and days; single 10-min values stay uncertain, so report a band.

In [9]:
scales = {"10 min": None, "1 hour": "1h", "3 hours": "3h", "1 day": "1D"}

def corr_at(a, b, rule):
    d = pd.concat([a, b], axis=1)
    d = d.resample(rule).mean() if rule else d
    return float(d.dropna().corr().iloc[0, 1])

res = {p: [corr_at(park["T"], park[p], rule) for rule in scales.values()] for p in "HO"}
K["agg"] = {p: [round(v, 3) for v in res[p]] for p in "HO"}
fig = go.Figure()
for p in "HO":
    fig.add_trace(go.Scatter(x=list(scales), y=res[p], mode="lines+markers", name=f"T vs park {p}",
                             line=dict(color=COL[p], width=2.5), marker=dict(size=8),
                             hovertemplate="%{x}: r = %{y:.3f}<extra>T vs " + p + "</extra>"))
    up = 14 if p == "O" else -16
    for i, sh in ((0, up), (-1, -up)):
        fig.add_annotation(x=list(scales)[i], y=res[p][i], text=f"{res[p][i]:.2f}", showarrow=False, yshift=sh,
                           font=dict(color=COL[p], size=12))
fig.update_xaxes(title="Averaging window applied to both parks", type="category")
fig.update_yaxes(title="Correlation r", range=[min(min(res["H"]), min(res["O"])) - 0.03, 1.0])
style(fig, f"Averaging removes noise: r rises from {res['H'][0]:.2f} to {res['H'][-1]:.2f} (T vs H)",
      "Correlation of park-mean wind speed after averaging both series over longer windows", height=440)
fig.show()

### 6. Timing between parks hardly matters
**Takeaway:** The peak sits at +20 min for H (r = 0.806 vs 0.795 at zero shift) and 0 min for O (0.810); the gain is below 0.02.

**For the model:** Use same-time values first; lagged H values are an optional small extra.

In [10]:
shifts = np.arange(-6, 7)                                  # steps of 10 min
minutes = shifts * 10
def lagcorr(ref):
    # shift k: pair T(t) with reference(t + k*10 min); k > 0 uses a later reference value
    return [float(park["T"].corr(ref.shift(-k))) for k in shifts]
lag = {p: lagcorr(park[p]) for p in "HO"}
K["lag_peak"] = {p: (int(minutes[int(np.argmax(lag[p]))]), float(max(lag[p]))) for p in "HO"}
K["lag0"] = {p: float(lag[p][6]) for p in "HO"}

fig = go.Figure()
for p in "HO":
    fig.add_trace(go.Scatter(x=minutes, y=lag[p], mode="lines+markers", name=f"T vs park {p}",
                             line=dict(color=COL[p], width=2.5), marker=dict(size=6),
                             hovertemplate="shift %{x} min: r = %{y:.3f}<extra>T vs " + p + "</extra>"))
    m, r = K["lag_peak"][p]
    fig.add_trace(go.Scatter(x=[m], y=[r], mode="markers+text", showlegend=False, hoverinfo="skip",
                             marker=dict(color=COL[p], size=13, line=dict(color="white", width=2)),
                             text=[f"peak {r:.3f} at {m:+d} min".replace("+0 ", "0 ")], textposition="top center",
                             textfont=dict(color=COL[p], size=12)))
fig.add_vline(x=0, line_color=COL["grey"], line_width=1)
fig.update_layout(hovermode="x unified")
fig.update_xaxes(title="Time shift of the reference [min]", dtick=20)
fig.update_yaxes(title="Correlation with park T [r]")
style(fig, f"Timing hardly matters: best shift is {K['lag_peak']['H'][0]:+d} min (H) and {K['lag_peak']['O'][0]:+d} min (O)".replace("+0 ", "0 "),
      "Correlation of T(t) with reference(t + shift). Positive shift = reference value taken from later (wind reaches the reference after T)",
      height=450)
fig.show()

### 7a. Wind direction: both parks see the same wind
**Takeaway:** The wind most often comes from the west (270°): 19% of the time at H and 21% at T; the top three sectors cover 55% at H.

In [11]:
centers = np.arange(0, 360, 30)
labels = ["N", "30°", "60°", "E", "120°", "150°", "S", "210°", "240°", "W", "300°", "330°"]
def sector(deg):
    return (((deg + 15) % 360) // 30).astype("Int64")
def share(deg):
    s = sector(deg.where(known)).dropna()
    return (s.value_counts(normalize=True).reindex(range(12), fill_value=0) * 100).values

share_T, share_H = share(dir_T), share(dir_H)
K["dom_T"], K["dom_T_pct"] = int(centers[share_T.argmax()]), float(share_T.max())
K["dom_H"], K["dom_H_pct"] = int(centers[share_H.argmax()]), float(share_H.max())
K["dom3_H"] = float(np.sort(share_H)[-3:].sum())

fig = go.Figure()
for name, sh, c in (("Park H", share_H, COL["H"]), ("Park T", share_T, COL["T"])):
    fig.add_trace(go.Barpolar(theta=centers, r=sh, width=28, name=name, marker=dict(color=c, line=dict(color="white", width=1)),
                              opacity=0.7, hovertemplate="from %{theta}°: %{r:.1f}% of time<extra>" + name + "</extra>"))
fig.update_layout(polar=dict(barmode="overlay", 
    angularaxis=dict(rotation=90, direction="clockwise", tickvals=centers, ticktext=labels, gridcolor="#ececea"),
    radialaxis=dict(ticksuffix="%", gridcolor="#ececea", angle=112, tickvals=[10, 20], tickfont=dict(size=11))))
style(fig, f"Wind blows from the west most often ({K['dom_H']}°, about {K['dom_H_pct']:.0f}% of the time), at both parks",
      "Share of 10-min readings by direction the wind comes from (12 sectors of 30°, 0° = north)", height=520)
fig.update_layout(margin=dict(t=100, b=40))
fig.show()

### 7b. Wind direction barely changes how T relates to H
**Takeaway:** T runs at 0.91 of H speed overall; by sector it ranges 0.85 (0°) to 0.95 (90°), at least 357 samples per sector.

**For the model:** Direction is a small correction, not a core input; try sector-specific ratios later.

In [12]:
d = pd.DataFrame({"sec": sector(dir_H.where(known)), "T": park["T"], "H": park["H"]}).dropna()
g = d.groupby("sec").agg(T=("T", "mean"), H=("H", "mean"), n=("T", "size"))
ratio = (g["T"] / g["H"]).reindex(range(12))
overall = d["T"].mean() / d["H"].mean()
K["ratio_overall"] = float(overall)
K["ratio_min"], K["ratio_max"] = float(ratio.min()), float(ratio.max())
K["ratio_min_dir"], K["ratio_max_dir"] = int(centers[int(ratio.idxmin())]), int(centers[int(ratio.idxmax())])
K["ratio_n_min"] = int(g["n"].min())

fig = go.Figure(go.Bar(x=[f"{c}°" for c in centers], y=ratio.values, marker_color=COL["T"],
                       text=[f"{v:.2f}" for v in ratio.values], textposition="inside", insidetextanchor="start", textfont=dict(size=12, color="white"),
                       customdata=g["n"].reindex(range(12)).values,
                       hovertemplate="H wind from %{x}<br>T / H = %{y:.2f}<br>%{customdata} samples<extra></extra>",
                       name="T / H ratio", showlegend=False))
fig.add_trace(go.Scatter(x=[f"{c}°" for c in centers], y=[overall] * 12, mode="lines", line=dict(color=COL["ink"], width=2),
                         name=f"Overall ratio {overall:.2f}", hovertemplate="overall %{y:.2f}<extra></extra>"))
fig.update_xaxes(title="Wind direction at park H, sector centre [° from north]")
fig.update_yaxes(title="Mean wind speed T / H [ratio]", range=[0, float(ratio.max()) * 1.18])
style(fig, f"T runs at {K['ratio_overall']:.2f} of H speed; direction shifts it by only {K['ratio_min']:.2f}–{K['ratio_max']:.2f}",
      "Mean T-park wind speed divided by mean H-park wind speed, by direction sector of H", height=440)
fig.show()

### 8. Wind dips mid-morning; T gets relatively more wind in the afternoon
**Takeaway:** H wind is lowest at 9:00 (4.9 m/s) and highest at 23:00 (6.7 m/s); T/H swings 0.85 (night) to 1.00 (15:00) around 0.91, wider than the direction effect (0.85–0.95).

**For the model:** Hour of day is the most promising secondary correction; add it as an hourly T/H factor and test on validation.

In [13]:
hr = park.dropna()                                          # steps where all three parks report
byh = hr.groupby(hr.index.hour).mean()
rh = byh["T"] / byh["H"]
K["hour_peak"], K["hour_low"] = int(byh["H"].idxmax()), int(byh["H"].idxmin())
K["hour_H_max"], K["hour_H_min"] = float(byh["H"].max()), float(byh["H"].min())
K["hr_ratio_min"], K["hr_ratio_max"] = float(rh.min()), float(rh.max())
K["hr_ratio_overall"] = float(hr["T"].mean() / hr["H"].mean())
K["hr_ratio_max_h"], K["hr_ratio_min_h"] = int(rh.idxmax()), int(rh.idxmin())

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.09)
for p in "HOT":
    fig.add_trace(go.Scatter(x=byh.index, y=byh[p], mode="lines+markers", name=names[p], marker=dict(size=5),
                             line=dict(color=COL[p], width=2.5 if p == "T" else 1.8),
                             hovertemplate="%{y:.2f} m/s"), row=1, col=1)
fig.add_trace(go.Scatter(x=byh.index, y=rh, mode="lines+markers", name="Ratio T / H", marker=dict(size=5),
                         line=dict(color=COL["ink"], width=2), hovertemplate="%{y:.3f}"), row=2, col=1)
fig.add_hline(y=K["hr_ratio_overall"], line_color=COL["grey"], line_width=1, row=2, col=1, annotation_text=f"overall {K['hr_ratio_overall']:.2f}", annotation_position="top left", annotation_font=dict(color=COL["grey"], size=11))
fig.update_layout(hovermode="x unified")
fig.update_yaxes(title="Mean wind speed [m/s]", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title="T / H [ratio]", row=2, col=1)
fig.update_xaxes(title="Hour of day [h]", dtick=3, row=2, col=1)
style(fig, f"T gets relatively more wind in the afternoon (T/H {K['hr_ratio_min']:.2f} at night, {K['hr_ratio_max']:.2f} at {K['hr_ratio_max_h']}:00)",
      f"Mean wind speed by hour of day, Jul–Sep (H: {K['hour_H_min']:.1f}–{K['hour_H_max']:.1f} m/s); bottom: ratio T/H by hour",
      height=560)
fig.show()

### 9. Power curve: where the energy comes from
**Takeaway:** T wind is below cut-in 12% of the time (no energy); 97% of energy comes from 3–11.5 m/s and only 3% from rated power; 1 m/s of error moves power by 440 kW at the median 5.3 m/s (825 kW at 9 m/s, up to 908 kW).

**For the model:** Errors at 4–10 m/s matter most; judge the model in kW and kWh, not only in m/s.

In [14]:
pc = power_curve
Tall = pd.Series(ws.loc[known, TARGETS].to_numpy().ravel()).dropna()
P = wind_to_power(Tall.values, power_curve)
bins = np.floor(Tall.values).astype(int)
tot = pd.DataFrame({"bin": bins, "P": P})
pct_time = tot.groupby("bin").size().div(len(tot)).mul(100)
pct_energy = tot.groupby("bin")["P"].sum().div(tot["P"].sum()).mul(100)
bx = np.arange(0, 27)
pct_time, pct_energy = pct_time.reindex(bx, fill_value=0), pct_energy.reindex(bx, fill_value=0)

med = float(np.median(Tall))
slope = lambda v: float((wind_to_power(v + 1, power_curve) - wind_to_power(v - 1, power_curve)) / 2)
K["med_ws"], K["slope_med"], K["slope_9"] = med, slope(med), slope(9.0)
K["mean_ws"] = float(Tall.mean())
K["t_below_cutin"] = float((Tall < 3).mean() * 100)
K["t_rated"] = float((Tall >= 11.5).mean() * 100)
K["e_rated"] = float(tot.loc[Tall.values >= 11.5, "P"].sum() / tot["P"].sum() * 100)
ramp = (Tall.values >= 3) & (Tall.values < 11.5)
K["t_ramp"], K["e_ramp"] = float(ramp.mean() * 100), float(tot.loc[ramp, "P"].sum() / tot["P"].sum() * 100)
K["cf"] = float(np.nanmean(P) / 4500 * 100)
K["slope_max"] = float(max(slope(v) for v in np.arange(3.5, 11, 0.5)))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45], vertical_spacing=0.1)
fig.add_trace(go.Scatter(x=list(pc.wind_speed) + [26.01], y=list(pc.power) + [0], mode="lines", name="Power curve",
                         line=dict(color=COL["T"], width=3), hovertemplate="%{x:.1f} m/s: %{y:,.0f} kW<extra></extra>"), row=1, col=1)
notes = [(3, 31, "cut-in 3 m/s", -30, -40), (11.5, 4500, "rated 4,500 kW from 11.5 m/s", -10, -35),
         (20.5, 4455, "derating from 20.5 m/s", 10, -40), (26, 2529, "cut-out 26 m/s (assumed)", -80, 70)]
for x, y, txt, ax, ay in notes:
    fig.add_annotation(x=x, y=y, text=txt, showarrow=True, arrowhead=2, ax=ax, ay=ay, xref="x", yref="y",
                       font=dict(size=12, color=COL["grey"]), arrowcolor=COL["grey"])
fig.add_trace(go.Bar(x=bx + 0.5, y=pct_time.values, width=0.4, name="% of time", marker_color=COL["T_light"],
                     offset=-0.4, hovertemplate="%{x:.1f} m/s bin: %{y:.1f}% of time<extra></extra>"), row=2, col=1)
fig.add_trace(go.Bar(x=bx + 0.5, y=pct_energy.values, width=0.4, name="% of energy", marker_color=COL["T"],
                     offset=0, hovertemplate="%{x:.1f} m/s bin: %{y:.1f}% of energy<extra></extra>"), row=2, col=1)
fig.update_layout(barmode="overlay")
fig.update_yaxes(title="Power per turbine [kW]", range=[0, 5200], row=1, col=1)
fig.update_yaxes(title="Share of total [%]", row=2, col=1)
fig.update_xaxes(range=[0, 27], dtick=2, row=1, col=1)
fig.update_xaxes(title="Wind speed [m/s] (bottom: 1 m/s bins)", range=[0, 27], dtick=2, row=2, col=1)
style(fig, f"{K['e_ramp']:.0f}% of energy comes from the steep part of the curve, where 1 m/s error ≈ {K['slope_med']:,.0f} kW",
      f"Top: turbine power curve. Bottom: actual T wind speeds (non-hidden) through the curve; median {med:.1f} m/s", height=640)
fig.show()

### 10a. Which inputs relate to T wind speed
**Takeaway:** Neighbour wind speed leads (O 0.81, best single reference O1 0.81, H 0.80); H stddev (0.70) mostly repeats wind speed; pressure 0.24, temperature 0.15, hour sin 0.06 and cos 0.12. Park O pressure and stddev are placeholders and unusable.

In [15]:
Tm = park["T"]
ref_r = ws[REFERENCES].corrwith(Tm).abs()
best_ref = ref_r.idxmax()
K["best_ref"], K["best_ref_r"] = best_ref, float(ref_r.max())
hour_frac = ws.index.hour + ws.index.minute / 60
feats = {
    "H-park mean wind speed": (park["H"], "H"),
    "O-park mean wind speed": (park["O"], "O"),
    f"Best single reference ({best_ref})": (ws[best_ref], best_ref[0]),
    "H wind-speed stddev": (sd[H_COLS].mean(axis=1), "H"),
    "H temperature": (temp[H_COLS].mean(axis=1), "H"),
    "H air pressure": (pres[H_COLS].mean(axis=1), "H"),
    "H direction (sin)": (np.sin(np.deg2rad(dir_H)), "H"),
    "H direction (cos)": (np.cos(np.deg2rad(dir_H)), "H"),
    "Hour of day (sin)": (pd.Series(np.sin(2 * np.pi * hour_frac / 24), index=ws.index), "x"),
    "Hour of day (cos)": (pd.Series(np.cos(2 * np.pi * hour_frac / 24), index=ws.index), "x"),
}
# simple model T ~ a + b*H + c*O, fitted on train rows only
X = pd.DataFrame({"c": 1.0, "H": park["H"], "O": park["O"]})
ok = train & X.notna().all(axis=1) & Tm.notna()
beta, *_ = np.linalg.lstsq(X[ok].values, Tm[ok].values, rcond=None)
resid = (Tm - X.values @ beta).where(train)
K["beta_park"] = [float(b) for b in beta]

group_name = {"H": "Park H input", "O": "Park O input", "x": "Time of day"}
group_col = {"H": COL["H"], "O": COL["O"], "x": COL["grey"]}

def bar_chart(values, title, subtitle, xtitle, exclude=()):
    items = sorted([(k, v, g) for k, (v, g) in values.items() if k not in exclude], key=lambda t: abs(t[1]))
    order = [k for k, _, _ in items]
    fig = go.Figure()
    for g in ("H", "O", "x"):
        sel = [(k, v) for k, v, gg in items if gg == g]
        if not sel:
            continue
        fig.add_trace(go.Bar(y=[k for k, _ in sel], x=[abs(v) for _, v in sel], orientation="h", name=group_name[g],
                             marker_color=group_col[g], text=[f"{abs(v):.2f}" for _, v in sel], textposition="outside",
                             customdata=[v for _, v in sel],
                             hovertemplate="%{y}<br>r = %{customdata:+.3f}<extra></extra>"))
    fig.update_yaxes(categoryorder="array", categoryarray=order, showgrid=False)
    fig.update_xaxes(title=xtitle, range=[0, max(abs(v) for _, v, _ in items) * 1.15], rangemode="tozero")
    style(fig, title, subtitle, height=440)
    fig.update_layout(margin=dict(l=230, t=100))
    return fig

corr_T = {k: (Tm.corr(v), g) for k, (v, g) in feats.items()}
corr_R = {k: (resid.corr(v), g) for k, (v, g) in feats.items()}
K["c_T"] = {k: round(float(v[0]), 3) for k, v in corr_T.items()}
K["c_R"] = {k: round(float(v[0]), 3) for k, v in corr_R.items()}
fig = bar_chart(corr_T, f"Neighbour wind speed dominates (|r| = {abs(corr_T['H-park mean wind speed'][0]):.2f}–{abs(corr_T['O-park mean wind speed'][0]):.2f}); the rest is weak",
                "Absolute correlation of each candidate input with the T-park mean wind speed (Jul–Sep, 10-min)",
                "Absolute correlation |r| with T-park wind speed")
fig.show()

### 10b. Do the extras add anything beyond the neighbours?
**Takeaway:** After T = a + b·H + c·O (0.67 + 0.39·H + 0.40·O), no extra exceeds |r| = 0.08 with the residual; stddev drops from 0.70 to 0.07.
Hour of day: sin 0.07, cos 0.03; the 15:00 peak is a ratio effect that these two smooth waves capture only partly.

**For the model:** Start with wind speeds; try an hourly T/H factor first, then direction, only if validation improves.

In [16]:
extras = ["H-park mean wind speed", "O-park mean wind speed"]
K["resid_top"] = max(((k, abs(v[0])) for k, v in corr_R.items() if k not in extras), key=lambda t: t[1])
K["resid_top"] = [K["resid_top"][0], float(K["resid_top"][1])]
fig = bar_chart(corr_R, f"Beyond the neighbours, extras add little (largest |r| = {K['resid_top'][1]:.2f})",
                "Absolute correlation with what the model T = a + b·H + c·O (fitted on train) gets wrong",
                "Absolute correlation |r| with the model residual", exclude=extras)
fig.show()

### 11a. Baseline accuracy on the validation window (4–9 Sep)
**Takeaway:** Park averages (A) give wind RMSE 1.40 m/s and power RMSE 878 kW; all 7 references (B) give 1.37 m/s and 862 kW, a gain of only 2% (same 864 steps per row).

**For the model:** Prefer A: nearly as accurate and explainable with three numbers.

In [17]:
const = pd.Series(1.0, index=ws.index, name="const")
FEATS = {"A: park averages": pd.concat([const, park[["H", "O"]]], axis=1),
         "B: all 7 references": pd.concat([const, ws[REFERENCES]], axis=1)}

def fit(feat, y):
    """Least squares on train rows; returns coefficients, prediction (>= 0) and residual std."""
    complete = feat.notna().all(axis=1)
    ok = train & complete & y.notna()
    beta, *_ = np.linalg.lstsq(feat[ok].values, y[ok].values, rcond=None)
    pred = pd.Series(feat.values @ beta, index=feat.index).where(complete).clip(lower=0)
    return beta, pred, float((y[ok] - feat[ok].values @ beta).std())

fits = {(m, t): fit(f, ws[t]) for m, f in FEATS.items() for t in TARGETS}
rows = []
for t in TARGETS:
    y = ws[t]
    common = val & y.notna() & fits[("A: park averages", t)][1].notna() & fits[("B: all 7 references", t)][1].notna()
    Pa = wind_to_power(y[common], power_curve)
    for m in FEATS:
        pred = fits[(m, t)][1][common]
        Pp = wind_to_power(pred, power_curve)
        e = pred - y[common]
        rows.append({"Target": t, "Model": m, "Steps": int(common.sum()),
                     "WS RMSE [m/s]": np.sqrt((e ** 2).mean()), "WS MAE [m/s]": e.abs().mean(),
                     "Power RMSE [kW]": np.sqrt(((Pp - Pa) ** 2).mean()), "Power MAE [kW]": np.abs(Pp - Pa).mean(),
                     "Energy deviation [%]": (energy_kwh(Pp) - energy_kwh(Pa)) / energy_kwh(Pa) * 100})
tab = pd.DataFrame(rows)
fmt = {c: "{:.2f}" for c in tab.columns[3:]}
fmt["Power RMSE [kW]"] = fmt["Power MAE [kW]"] = "{:.0f}"
display(tab.style.hide(axis="index").format(fmt))
mean_tab = tab.groupby("Model")[tab.columns[3:]].mean()
K["tab"] = {m: {c: float(v) for c, v in r.items()} for m, r in mean_tab.iterrows()}
K["tab_rows"] = [{k: (v if isinstance(v, str) else float(v)) for k, v in r.items()} for r in rows]
K["val_steps"] = int(tab["Steps"].iloc[0])
K["beta_A"] = {t: [float(b) for b in fits[("A: park averages", t)][0]] for t in TARGETS}

Target,Model,Steps,WS RMSE [m/s],WS MAE [m/s],Power RMSE [kW],Power MAE [kW],Energy deviation [%]
T1,A: park averages,864,1.60,1.29,970,714,18.59
T1,B: all 7 references,864,1.55,1.24,937,679,17.21
T2,A: park averages,864,1.25,0.95,792,539,-5.40
T2,B: all 7 references,864,1.24,0.94,787,534,-4.04
T3,A: park averages,864,1.34,1.04,874,621,-8.16
T3,B: all 7 references,864,1.33,1.02,863,612,-7.11


### 11b. Predicted vs actual wind speed at T1
**Takeaway:** The prediction follows the shape of T1 (σ = 1.36 m/s), but only 89% of actual values lie inside the 95% band, so the band is too narrow here.

**For the model:** Widen the band (errors are correlated in time) or calibrate it on the validation window.

In [18]:
mA, t1 = "A: park averages", "T1"
beta1, pred1, sigma1 = fits[(mA, t1)]
vt = ws.index[val]
y1, p1 = ws.loc[vt, t1], pred1.loc[vt]
band = 1.96 * sigma1
inside = float(((y1 - p1).abs() <= band)[y1.notna() & p1.notna()].mean() * 100)
K["sigma_T1"], K["band_T1"], K["cover_T1"] = sigma1, band, inside

fig = go.Figure()
fig.add_trace(go.Scatter(x=vt, y=p1 - band, mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=vt, y=p1 + band, mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(138,137,132,0.22)", name="±1.96σ band (95 %)", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=vt, y=y1, mode="lines", name="Actual T1", line=dict(color=COL["T"], width=2),
                         hovertemplate="%{y:.1f} m/s"))
fig.add_trace(go.Scatter(x=vt, y=p1, mode="lines", name="Model A prediction", line=dict(color=COL["ink"], width=1.5),
                         hovertemplate="%{y:.1f} m/s"))
fig.update_layout(hovermode="x unified")
fig.update_xaxes(title="Time (validation window, 4–9 Sep)")
fig.update_yaxes(title="Wind speed at T1 [m/s]", rangemode="tozero")
style(fig, f"Prediction follows T1, but only {inside:.0f}% of values fall inside the 95% band (band too narrow)",
      f"Validation window, T1 wind speed. Band = prediction ± 1.96 × {sigma1:.2f} m/s (training residual spread)", height=470)
fig.show()

### 11c. How one prediction is built
**Takeaway:** At 04 Sep 05:00: 0.80 + 0.39 × 6.3 + 0.37 × 7.0 = 5.83 m/s, which the power curve turns into 966 kW (actual 7.7 m/s = 2,290 kW).

**For the model:** Each step is a number a non-expert can follow; the power curve is applied last.

In [19]:
a, b, c = fits[(mA, t1)][0]
pr = pd.DataFrame({"H": park["H"], "O": park["O"], "y": ws[t1]}).loc[vt].dropna()
pr["pred"] = a + b * pr["H"] + c * pr["O"]
ts = (pr["pred"] - pr["y"].median()).abs().idxmin()                 # a typical moment in the window
h_, o_, y_, p_ = pr.loc[ts, ["H", "O", "y", "pred"]]
P_pred, P_act = float(wind_to_power(p_, power_curve)), float(wind_to_power(y_, power_curve))
K["wf"] = dict(ts=f"{ts:%d %b %H:%M}", a=float(a), b=float(b), c=float(c), h=float(h_), o=float(o_),
               pred=float(p_), y=float(y_), P_pred=P_pred, P_act=P_act)

steps = [("Starting value (intercept)", 0, a, COL["grey"]),
         (f"H-park mean {h_:.1f} m/s × {b:.2f}", a, b * h_, COL["H"]),
         (f"O-park mean {o_:.1f} m/s × {c:.2f}", a + b * h_, c * o_, COL["O"]),
         ("Predicted T1 speed", 0, p_, COL["T"])]
fig = go.Figure(go.Bar(x=[t[0] for t in steps], y=[t[2] for t in steps], base=[t[1] for t in steps],
                       marker_color=[t[3] for t in steps], text=[f"{a:+.2f}", f"{b * h_:+.2f}", f"{c * o_:+.2f}", f"{p_:.2f}"],
                       textposition="outside", hovertemplate="%{x}: %{y:.2f} m/s<extra></extra>"))
fig.update_yaxes(title="Wind speed contribution [m/s]", rangemode="tozero")
fig.update_xaxes(title=f"Example: {ts:%d %b %Y, %H:%M}")
style(fig, f"One prediction = {a:.2f} + {b:.2f} × H + {c:.2f} × O, then the power curve",
      f"T1 at {ts:%d %b %H:%M}: {p_:.1f} m/s predicted gives {P_pred:,.0f} kW (actual {y_:.1f} m/s gives {P_act:,.0f} kW)",
      height=440)
fig.show()

### 11c+. Where a, b, c come from, and what stays fixed
**How they are set:** Ordinary least squares, once per turbine. Take every 10-min step in the training period (all data except 1–3 Sep and 4–9 Sep) where T, H and O all have a value, and pick the three numbers that minimise the squared error of T = a + b·H + c·O. Nothing is tuned by hand, and the validation window is never used for the fit.

**Static vs varying:**
- **Fixed in time:** one set per turbine, reused for every 10-min step, including the hidden 1–3 Sep. The model does not learn from the hours just before the outage.
- **Per turbine:** b and c are almost equal (0.39/0.37 for T1, 0.41/0.44 for T2, 0.39/0.38 for T3); only a differs (0.80, 0.50, 0.72 m/s). Note that b + c is below 1: the fit shrinks towards the mean because the references are an imperfect copy of T.
- **Not the same in every month:** fitted on one month only, a falls from 1.23 (Jul) to 0.27 (Sep) and b rises from 0.32 to 0.57. The relation drifts with the season, so use recent data.
- **Barely different by time of day:** separate fits per 6 h block move validation RMSE from 1.60 to 1.60 m/s.
- **Slightly better by wind direction:** separate fits per quadrant move RMSE from 1.60 to 1.56 m/s (2% gain).


**For the model:** Keep three static numbers per turbine as the explainable core. A direction-specific or recent-weighted fit is the next step to try, because the month drift is larger than the gain from time of day.


In [20]:
# Same recipe (least squares on train rows), but fitted separately per group, to see what is static and what varies
Xa = FEATS[mA]
hour = pd.Series(ws.index.hour, index=ws.index)
groups = {
    "Turbine": {t: None for t in TARGETS},
    "Month (T1)": {"Jul": ws.index.month == 7, "Aug": ws.index.month == 8, "Sep": ws.index.month == 9},
    "Time of day (T1)": {"00–06": hour < 6, "06–12": (hour >= 6) & (hour < 12),
                         "12–18": (hour >= 12) & (hour < 18), "18–24": hour >= 18},
    "Wind from, H park (T1)": {"N": (dir_H >= 315) | (dir_H < 45), "E": (dir_H >= 45) & (dir_H < 135),
                               "S": (dir_H >= 135) & (dir_H < 225), "W": (dir_H >= 225) & (dir_H < 315)},
}
rows, seg_gain = [], {}
for kind, grp in groups.items():
    for name, mask in grp.items():
        t = name if kind == "Turbine" else "T1"
        m = pd.Series(True, index=ws.index) if mask is None else pd.Series(np.asarray(mask), index=ws.index)
        ok = train & m & Xa.notna().all(axis=1) & ws[t].notna()
        beta, *_ = np.linalg.lstsq(Xa[ok].values, ws[t][ok].values, rcond=None)
        rows.append({"Fitted separately for": kind, "Group": name, "a": beta[0], "b (H)": beta[1], "c (O)": beta[2],
                     "Train steps": int(ok.sum())})
coef = pd.DataFrame(rows)
display(coef.style.hide(axis="index").format({"a": "{:.2f}", "b (H)": "{:.2f}", "c (O)": "{:.2f}"}))

# does a group-specific fit beat the single static fit on the validation window (T1)?
y = ws["T1"]; base = fits[(mA, "T1")][1]
okv = val & y.notna() & base.notna()
rm = lambda p: float(np.sqrt(((p[okv] - y[okv]) ** 2).mean()))
for kind in ["Time of day (T1)", "Wind from, H park (T1)"]:
    pred = base.copy()
    for name, mask in groups[kind].items():
        r = coef[(coef["Fitted separately for"] == kind) & (coef.Group == name)].iloc[0]
        mm = pd.Series(np.asarray(mask), index=ws.index)
        pred[mm] = (r["a"] + r["b (H)"] * Xa["H"] + r["c (O)"] * Xa["O"])[mm].clip(lower=0)
    seg_gain[kind] = (rm(base), rm(pred))
K["coef"] = coef.to_dict("records")
K["seg_gain"] = {k: [float(x) for x in v] for k, v in seg_gain.items()}
K["static_rmse_T1"] = rm(base)
for k, (s, g) in seg_gain.items():
    print(f"{k}: static {s:.3f} m/s vs separate fits {g:.3f} m/s on validation")

Fitted separately for,Group,a,b (H),c (O),Train steps
Turbine,T1,0.80,0.39,0.37,11508
Turbine,T2,0.50,0.41,0.44,11555
Turbine,T3,0.72,0.39,0.38,11553
Month (T1),Jul,1.23,0.32,0.35,4417
Month (T1),Aug,0.75,0.35,0.43,4101
Month (T1),Sep,0.27,0.57,0.28,2990
Time of day (T1),00–06,1.65,0.30,0.31,2889
Time of day (T1),06–12,0.57,0.45,0.30,2880
Time of day (T1),12–18,0.74,0.37,0.43,2873
Time of day (T1),18–24,0.77,0.40,0.38,2866


Time of day (T1): static 1.603 m/s vs separate fits 1.597 m/s on validation
Wind from, H park (T1): static 1.603 m/s vs separate fits 1.565 m/s on validation


### 11d. Energy: predicted vs actual
**Takeaway:** The park total is within 0.1% only because errors cancel. Over 4–9 Sep the three turbines together are 798 MWh predicted vs 797 MWh actual (+0.1%), but T1 is +19%, T2 -5%, T3 -8%; averaging the power over the ±σ spread shifts the total to +2.6%.

**For the model:** Per-turbine bias needs a check (T1 reads lower than its neighbours in this window); report park totals first.

In [21]:
sel = tab[tab.Model == mA].set_index("Target")
act, prd = {}, {}
for t in TARGETS:
    common = val & ws[t].notna() & fits[(mA, t)][1].notna()
    act[t] = energy_kwh(wind_to_power(ws[t][common], power_curve)) / 1000
    prd[t] = energy_kwh(wind_to_power(fits[(mA, t)][1][common], power_curve)) / 1000
dev = {t: (prd[t] - act[t]) / act[t] * 100 for t in TARGETS}
K["mwh"] = {t: [act[t], prd[t], dev[t]] for t in TARGETS}
K["mwh_total"] = [sum(act.values()), sum(prd.values()), (sum(prd.values()) - sum(act.values())) / sum(act.values()) * 100]

# effect of the power curve on uncertainty: average the power over the +-sigma spread of the wind prediction
z, w = np.polynomial.hermite_e.hermegauss(21)
w = w / w.sum()
ep_total = 0.0
for t in TARGETS:
    common = val & ws[t].notna() & fits[(mA, t)][1].notna()
    sg = fits[(mA, t)][2]
    pr_t = fits[(mA, t)][1][common].values
    ep_total += sum(wk_ * energy_kwh(wind_to_power(np.clip(pr_t + sg * zk, 0, None), power_curve)) for zk, wk_ in zip(z, w)) / 1000
K["ep_dev"] = (ep_total - sum(act.values())) / sum(act.values()) * 100

fig = go.Figure()
fig.add_trace(go.Bar(x=TARGETS, y=[act[t] for t in TARGETS], name="Actual", marker_color=COL["T"],
                     hovertemplate="%{x} actual: %{y:.1f} MWh<extra></extra>"))
fig.add_trace(go.Bar(x=TARGETS, y=[prd[t] for t in TARGETS], name="Model A prediction", marker_color=COL["T_light"],
                     text=[f"{dev[t]:+.1f}%" for t in TARGETS], textposition="outside",
                     hovertemplate="%{x} predicted: %{y:.1f} MWh<extra></extra>"))
fig.update_layout(barmode="group")
fig.update_yaxes(title="Energy in validation window [MWh]", rangemode="tozero")
fig.update_xaxes(title="Turbine")
style(fig, f"Park total is within {abs(K['mwh_total'][2]):.1f}% only because errors cancel: T1 {dev['T1']:+.0f}%, T2 {dev['T2']:+.0f}%, T3 {dev['T3']:+.0f}%",
      f"Energy 4–9 Sep from actual vs predicted wind speed through the power curve; labels = deviation of prediction from actual", height=440)
fig.show()

### 12. From data to explainable model

| Input | Evidence | How to use it | How to explain it to a non-data person |
|---|---|---|---|
| Neighbour wind speed (park averages) | r 0.80 (H), 0.81 (O); RMSE 1.40 m/s | Fit T = a + b·H + c·O per turbine | Neighbouring parks feel the same weather, a bit stronger. |
| Multiple references | All 7 vs park averages: RMSE 1.37 vs 1.40 m/s | Keep the 3-number model; use all 7 as a check | More neighbours barely help, so keep it simple. |
| Time lag | Best shift +20 min (H); r gain 0.011 | Use same-time values | Weather arrives at all parks at nearly the same time. |
| Wind direction sectors | T/H ratio 0.85–0.95 by sector | Optional sector-wise scale factor | Wind direction changes T's share of H only slightly. |
| Hour of day | T/H 0.85 to 1.00; residual r sin 0.07, cos 0.03 | Best secondary fix: hourly T/H factor | Afternoons give T a slightly larger share of H's wind. |
| Temperature, pressure, stddev | Residual r at most 0.07 | Leave out; park O has none | They add nothing beyond the neighbours' wind speed. |
| Power curve / E[P] effect | 440 kW per m/s at median; E[P] shifts energy to +2.6% | Apply curve last; test E[P] over σ | Small wind errors turn into larger power errors in the steep middle. |
| Uncertainty band | σ = 1.36 m/s; 89% of values inside 95% band | Report band; widen it if needed | We give a range, not a single number, for every estimate. |